# Main Pipeline Benchmark Notebook

This notebook mirrors the active `main.py` experiment flow in a notebook format.

Pipeline:
1. Load data
2. Create sliding windows
3. Train all registered models from the current factory
4. Generate samples
5. Compute the same evaluation metrics
6. Plot and compare the models


In [1]:
import os
import sys
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from IPython.display import display

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'main.py').exists():
    sys.path.insert(0, str(PROJECT_ROOT))
else:
    PROJECT_ROOT = Path('/home/wan/psc/sbts_advanced')
    sys.path.insert(0, str(PROJECT_ROOT))

from data.loaders import load_etf_data, load_synthetic_data, create_sliding_windows
from models.factory import get_model, get_default_config, list_models
from metrics import discriminative_score_metrics, predictive_score_metrics
from metrics.numba_metrics import compute_all_metrics_numba, compute_stylized_facts_numba
from visualization import set_style

warnings.filterwarnings('ignore')
set_style()


## Configuration

By default this notebook runs all models currently registered in `models.factory`.
If the full run is too heavy, replace `MODELS_TO_RUN` with a smaller subset.


In [2]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

USE_SYNTHETIC = True
MODELS_TO_RUN = list(list_models().keys())
# Example lighter subset:
# MODELS_TO_RUN = ['jd_sbts', 'jd_sbts_f', 'lightsb', 'timegan', 'diffusion_ts', 'rnn', 'transformer_ar']

DATA_CONFIG = {
    'tickers': ['SPY', 'QQQ', 'IWM', 'EEM', 'GLD'],
    'start_date': '2020-01-01',
    'end_date': '2024-01-01',
    'window_size': 60,
    'stride': 5,
    'synthetic_type': 'gbm_jump',
    'n_synthetic_samples': 64,
    'synthetic_total_steps': 240,
    'synthetic_return_type': 'log_returns',
}

TRAINING_OVERRIDES = {
    'seed': SEED,
    'verbose': False,
    'lstm_epochs': 25,
    'timegan_epochs': 20,
    'diffusion_epochs': 25,
    'lightsb_epochs': 25,
    'rnn_epochs': 25,
    'transformer_ar_epochs': 25,
    'n_generate': 64,
    'acf_max_lag': 15,
    'discriminative_iterations': 200,
    'predictive_iterations': 200,
}

FAIR_COMPARISON = {
    'enabled': True,
    'shared_eval_subset': True,
    'shared_conditioning': True,
    'reset_seed_per_model': True,
}

OUTPUT_DIR = PROJECT_ROOT / 'notebook_outputs' / 'benchmark_main_pipeline'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Models to run:', MODELS_TO_RUN)
print('Output dir:', OUTPUT_DIR)
print('Fair comparison settings:', FAIR_COMPARISON)


Models to run: ['jd_sbts', 'jd_sbts_f', 'jd_sbts_neural', 'jd_sbts_f_neural', 'lightsb', 'numba_sb', 'timegan', 'diffusion_ts', 'rnn', 'transformer_ar']
Output dir: /home/wan/psc/sbts_advanced/notebook_outputs/benchmark_main_pipeline


## Load Data And Create Sliding Windows

In [3]:
if USE_SYNTHETIC:
    raw_data, _, metadata = load_synthetic_data(
        n_samples=DATA_CONFIG['n_synthetic_samples'],
        n_steps=DATA_CONFIG['synthetic_total_steps'],
        n_features=len(DATA_CONFIG['tickers']),
        data_type=DATA_CONFIG['synthetic_type'],
        seed=SEED,
        return_type=DATA_CONFIG['synthetic_return_type'],
    )
else:
    raw_data, _, metadata = load_etf_data(
        tickers=DATA_CONFIG['tickers'],
        start_date=DATA_CONFIG['start_date'],
        end_date=DATA_CONFIG['end_date'],
        return_type='log_returns',
    )

window_size = DATA_CONFIG['window_size']
stride = DATA_CONFIG['stride']

data = create_sliding_windows(raw_data, window_size=window_size, stride=stride)

time_grid = np.linspace(0, 1, data.shape[1])
n_generate = min(TRAINING_OVERRIDES['n_generate'], len(data))
shared_eval_subset = data[:n_generate].copy()

print('Metadata:', metadata)
print('Raw data shape:', raw_data.shape)
print('Windowed data shape:', data.shape)
print('Time grid length:', len(time_grid))
print('Samples for generation/evaluation:', n_generate)
print('Shared evaluation subset shape:', shared_eval_subset.shape)


Metadata: {'data_type': 'gbm_jump', 'n_samples': 240, 'n_steps': 60, 'n_features': 5, 'seed': 42}
Windowed data shape: (240, 60, 5)
Time grid length: 60
Samples for generation/evaluation: 64


## Train, Generate, Evaluate

In [4]:
def extract_metric_value(result, key, default):
    if isinstance(result, dict):
        return float(result.get(key, default))
    if np.isscalar(result):
        return float(result)
    return float(default)


def reset_generation_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


CONDITIONED_X0_MODELS = {'jd_sbts', 'jd_sbts_f', 'jd_sbts_neural', 'jd_sbts_f_neural'}
CONDITIONED_PREFIX_MODELS = {'rnn', 'transformer_ar'}
WINDOW_LEVEL_NOISE_MODELS = {'lightsb'}
UNCONDITIONAL_MODELS = {'timegan', 'diffusion_ts'}


def build_generation_kwargs(model_name, model, reference_windows, n_samples, n_steps):
    kwargs = {'n_samples': n_samples, 'n_steps': n_steps}
    protocol = 'shared_seed_only'
    note = 'Unconditional generation; fairness comes from shared training data, shared eval subset, and reset RNG seed.'

    if model_name in WINDOW_LEVEL_NOISE_MODELS:
        protocol = 'shared_seed_only'
        note = 'Window-level bridge model; its x0 is flattened latent noise for the full window, not a real first-step state.'

    if FAIR_COMPARISON['enabled'] and FAIR_COMPARISON['shared_conditioning']:
        if model_name in CONDITIONED_X0_MODELS:
            kwargs['x0'] = reference_windows[:, 0, :].astype(np.float32)
            protocol = 'shared_x0'
            note = 'All samples use the same real initial states from the shared evaluation subset.'
        elif model_name in CONDITIONED_PREFIX_MODELS:
            context_len = int(getattr(model, 'context_len', max(1, n_steps // 2)))
            context_len = max(1, min(context_len, reference_windows.shape[1] - 1))
            kwargs['x0'] = reference_windows[:, :context_len, :].astype(np.float32)
            protocol = f'shared_prefix_len_{context_len}'
            note = f'All samples use the same real prefix windows of length {context_len} from the shared evaluation subset.'

    return kwargs, protocol, note


trained_models = {}
generated_data = {}
stress_trajectories = {}
metrics_results = {}
training_times = {}
generation_times = {}
generation_protocols = {}
fairness_notes = {}
failures = {}

for model_name in MODELS_TO_RUN:
    print()
    print(f'=== {model_name} ===')
    try:
        config = get_default_config(model_name)
        config.update(TRAINING_OVERRIDES)
        config.setdefault('transformer_ar_max_seq_len', data.shape[1])
        config.setdefault('window_size', data.shape[1])
        config.setdefault('n_steps', data.shape[1])

        model = get_model(model_name, config)

        start = time.perf_counter()
        model.fit(data, time_grid, verbose=False)
        training_times[model_name] = time.perf_counter() - start
        trained_models[model_name] = model

        if FAIR_COMPARISON['enabled'] and FAIR_COMPARISON['reset_seed_per_model']:
            reset_generation_seed(SEED)

        gen_kwargs, protocol, note = build_generation_kwargs(
            model_name,
            model,
            shared_eval_subset,
            n_generate,
            data.shape[1],
        )
        generation_protocols[model_name] = protocol
        fairness_notes[model_name] = note

        start = time.perf_counter()
        if hasattr(model, 'use_feedback') and model.use_feedback:
            gen_paths, stress = model.generate(return_stress=True, **gen_kwargs)
            stress_trajectories[model_name] = stress
        else:
            gen_paths = model.generate(**gen_kwargs)
        generation_times[model_name] = time.perf_counter() - start
        generated_data[model_name] = gen_paths

        real_eval = shared_eval_subset[:min(len(shared_eval_subset), len(gen_paths))]
        gen_eval = gen_paths[:len(real_eval)]

        stats = compute_all_metrics_numba(real_eval, gen_eval, max_acf_lag=TRAINING_OVERRIDES['acf_max_lag'])
        gen_returns = np.diff(gen_eval, axis=1)
        if gen_returns.ndim == 3:
            gen_returns_1d = gen_returns[:, :, 0]
        else:
            gen_returns_1d = gen_returns
        stylized = compute_stylized_facts_numba(gen_returns_1d.flatten())

        disc = discriminative_score_metrics(
            real_eval,
            gen_eval,
            iterations=TRAINING_OVERRIDES['discriminative_iterations'],
        )
        pred = predictive_score_metrics(
            real_eval,
            gen_eval,
            iterations=TRAINING_OVERRIDES['predictive_iterations'],
        )

        metrics_results[model_name] = {
            'train_time': training_times[model_name],
            'gen_time': generation_times[model_name],
            'generation_protocol': protocol,
            'wasserstein_distance': stats.get('wasserstein_distance', np.nan),
            'acf_mse': stats.get('acf_mse', np.nan),
            'correlation_distance': stats.get('correlation_distance', np.nan),
            'discriminative_score': extract_metric_value(disc, 'discriminative_score', np.nan),
            'predictive_score': extract_metric_value(pred, 'predictive_score', np.nan),
            **{f'stylized_{k}': v for k, v in stylized.items()},
        }

        print(
            f"train_time={training_times[model_name]:.2f}s, "
            f"gen_time={generation_times[model_name]:.2f}s, "
            f"shape={gen_paths.shape}, protocol={protocol}"
        )
    except Exception as exc:
        failures[model_name] = str(exc)
        print(f'FAILED: {exc}')



=== jd_sbts ===
train_time=4.81s, gen_time=2.67s, shape=(64, 60, 5)

=== jd_sbts_f ===
train_time=3.21s, gen_time=2.01s, shape=(64, 60, 5)

=== jd_sbts_neural ===
  [Neural Jump] Epoch 10/30, Loss: 0.0035
  [Neural Jump] Epoch 20/30, Loss: 0.0029
  [Neural Jump] Epoch 30/30, Loss: 0.0013
train_time=11.48s, gen_time=0.06s, shape=(64, 60, 5)

=== jd_sbts_f_neural ===
  [Neural Jump] Epoch 10/30, Loss: 0.0034
  [Neural Jump] Epoch 20/30, Loss: 0.0026
  [Neural Jump] Epoch 30/30, Loss: 0.0009
train_time=12.50s, gen_time=0.07s, shape=(64, 60, 5)

=== lightsb ===
train_time=0.11s, gen_time=0.04s, shape=(64, 60, 5)

=== numba_sb ===
train_time=0.00s, gen_time=0.01s, shape=(64, 60, 5)

=== timegan ===
train_time=0.30s, gen_time=0.00s, shape=(64, 60, 5)

=== diffusion_ts ===
train_time=0.14s, gen_time=0.09s, shape=(64, 60, 5)

=== rnn ===
train_time=0.06s, gen_time=0.01s, shape=(64, 60, 5)

=== transformer_ar ===
train_time=0.34s, gen_time=0.11s, shape=(64, 60, 5)


## Metrics Summary

In [5]:
df_metrics = pd.DataFrame(metrics_results).T.sort_index()
display(df_metrics)

if generation_protocols:
    protocol_df = pd.DataFrame({
        'generation_protocol': pd.Series(generation_protocols),
        'fairness_note': pd.Series(fairness_notes),
    }).sort_index()
    print('Generation protocol summary')
    display(protocol_df)

if failures:
    print('Failures:')
    display(pd.Series(failures, name='error'))


,train_time,gen_time,wasserstein_distance,acf_mse,correlation_distance,discriminative_score,predictive_score,stylized_volatility_clustering,stylized_fat_tails,stylized_leverage_effect
diffusion_ts,0.141038,0.089596,1.687917,0.208494,0.729357,0.115385,100.962440,0.023555,0.007341,-0.002368
jd_sbts,4.808268,2.674070,4.428863,0.001985,0.712426,0.000000,101.469101,0.005027,0.227227,0.001021
jd_sbts_f,3.210164,2.005569,3.543948,0.002186,0.818608,0.192308,101.156433,0.000388,0.172352,-0.006610
jd_sbts_f_neural,12.495892,0.067333,4.115626,0.000416,1.039973,0.000000,101.720520,-0.004639,0.079999,-0.010885
jd_sbts_neural,11.480013,0.060854,4.409723,0.001215,0.608992,0.076923,102.052261,0.001205,0.033500,-0.002016
lightsb,0.109460,0.037816,4.167465,0.200845,0.706750,0.500000,101.876221,0.026649,0.050782,0.001419
numba_sb,0.000595,0.006160,1.099020,0.028585,0.764062,0.038462,101.925629,0.117287,0.618586,-0.003047
rnn,0.063718,0.014326,9.626633,0.002577,3.942325,0.000000,101.039108,0.164632,25.129094,0.029420
timegan,0.301277,0.001330,14.442632,0.000373,1.673404,0.000000,101.045166,0.252991,6.351994,0.510201
transformer_ar,0.341783,0.107002,0.851762,0.000306,1.392656,0.000000,102.169701,0.102548,0.655567,-0.006411


## Ranking And Decision Support


In [ ]:
MAIN_METRICS = ['wasserstein_distance', 'acf_mse', 'predictive_score']
SECONDARY_METRICS = ['correlation_distance', 'discriminative_score']
MAIN_WEIGHT = 0.7
SECONDARY_WEIGHT = 0.3

comparison_df = df_metrics.copy()
real_subset = shared_eval_subset.copy()
real_returns = np.diff(real_subset, axis=1)
if real_returns.ndim == 3:
    real_returns_1d = real_returns[:, :, 0]
else:
    real_returns_1d = real_returns
real_stylized = compute_stylized_facts_numba(real_returns_1d.flatten())

stylized_delta_cols = []
for fact_name, real_value in real_stylized.items():
    metric_col = f'stylized_{fact_name}'
    if metric_col in comparison_df.columns:
        delta_col = f'{metric_col}_delta'
        scale = max(abs(real_value), 1e-8)
        comparison_df[delta_col] = (comparison_df[metric_col] - real_value).abs() / scale
        stylized_delta_cols.append(delta_col)

if stylized_delta_cols:
    comparison_df['stylized_score'] = comparison_df[stylized_delta_cols].mean(axis=1)
else:
    comparison_df['stylized_score'] = np.nan

main_available = [col for col in MAIN_METRICS if col in comparison_df.columns]
secondary_available = [col for col in SECONDARY_METRICS if col in comparison_df.columns]
if comparison_df['stylized_score'].notna().any():
    secondary_available = secondary_available + ['stylized_score']

main_ranks = comparison_df[main_available].rank(method='min', ascending=True)
secondary_ranks = comparison_df[secondary_available].rank(method='min', ascending=True)

comparison_df['main_rank_mean'] = main_ranks.mean(axis=1)
comparison_df['secondary_rank_mean'] = secondary_ranks.mean(axis=1)
comparison_df['overall_rank_score'] = (
    MAIN_WEIGHT * comparison_df['main_rank_mean']
    + SECONDARY_WEIGHT * comparison_df['secondary_rank_mean']
)

rank_columns = [
    'generation_protocol',
    'main_rank_mean',
    'secondary_rank_mean',
    'overall_rank_score',
    *main_available,
    *secondary_available,
]
ranking_summary = comparison_df[rank_columns].sort_values(
    ['overall_rank_score', 'main_rank_mean', 'secondary_rank_mean']
)

print('Fair-comparison note')
print('- All models use the same training windows and the same shared evaluation subset.')
print('- Conditioned models receive shared initial conditions or shared prefixes where their API supports it.')
print('- Unconditional models only share the reset RNG seed and the shared evaluation subset.')

print('Main metrics: lower is better')
display(comparison_df[main_available].sort_values(main_available).head(len(comparison_df)))

print('Secondary metrics: lower is better')
display(comparison_df[secondary_available].sort_values(secondary_available).head(len(comparison_df)))

print('Real stylized-facts reference')
display(pd.Series(real_stylized, name='real_value').to_frame())

if stylized_delta_cols:
    print('Stylized-facts relative error: lower is better')
    display(comparison_df[stylized_delta_cols + ['stylized_score']].sort_values('stylized_score'))

print('Overall ranking summary')
display(ranking_summary)


## Main Metrics, Secondary Metrics, And Sanity Checks


In [ ]:
def plot_sorted_metric(ax, frame, column, title, color='steelblue'):
    series = frame[column].dropna().sort_values()
    series.plot(kind='bar', ax=ax, color=color)
    ax.set_title(title)
    ax.set_ylabel('Value')
    ax.tick_params(axis='x', rotation=45)


if generated_data:
    methods = list(generated_data.keys())
    sanity_subset = shared_eval_subset.copy()

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    main_plot_cols = [col for col in ['wasserstein_distance', 'acf_mse', 'predictive_score'] if col in comparison_df.columns]
    colors = ['steelblue', 'darkorange', 'seagreen']

    for ax, col, color in zip(axes.reshape(-1), main_plot_cols, colors):
        plot_sorted_metric(ax, comparison_df, col, col.replace('_', ' ').title(), color=color)

    ax = axes.reshape(-1)[len(main_plot_cols)]
    plot_sorted_metric(ax, comparison_df, 'overall_rank_score', 'Overall Rank Score', color='slateblue')

    for ax in axes.reshape(-1)[len(main_plot_cols) + 1:]:
        ax.axis('off')

    fig.suptitle('Primary Decision Metrics', fontsize=18, y=1.02)
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    secondary_plot_cols = [col for col in ['correlation_distance', 'discriminative_score', 'stylized_score'] if col in comparison_df.columns]
    colors = ['firebrick', 'teal', 'goldenrod']

    for ax, col, color in zip(axes.reshape(-1), secondary_plot_cols, colors):
        plot_sorted_metric(ax, comparison_df, col, col.replace('_', ' ').title(), color=color)

    ax = axes.reshape(-1)[len(secondary_plot_cols)]
    plot_sorted_metric(ax, comparison_df, 'secondary_rank_mean', 'Secondary Rank Mean', color='mediumpurple')

    for ax in axes.reshape(-1)[len(secondary_plot_cols) + 1:]:
        ax.axis('off')

    fig.suptitle('Secondary Metrics', fontsize=18, y=1.02)
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    ax = axes[0]
    ax.plot(np.mean(sanity_subset[:, :, 0], axis=0), color='black', linewidth=2.5, label='Real Mean')
    for method in methods:
        gen_subset = generated_data[method][:len(sanity_subset)]
        ax.plot(np.mean(gen_subset[:, :, 0], axis=0), linewidth=1.8, label=method)
    ax.set_title('Mean Path')
    ax.set_xlabel('Time Step')
    ax.legend(fontsize=8)

    ax = axes[1]
    sns.kdeplot(sanity_subset[:, -1, 0], ax=ax, color='black', fill=True, alpha=0.25, label='Real')
    for method in methods:
        gen_subset = generated_data[method][:len(sanity_subset)]
        sns.kdeplot(gen_subset[:, -1, 0], ax=ax, linewidth=2, label=method)
    ax.set_title('Terminal Distribution')
    ax.legend(fontsize=8)

    ax = axes[2]
    max_lag = TRAINING_OVERRIDES['acf_max_lag']
    series = sanity_subset[:, :, 0]
    real_acf = []
    for lag in range(max_lag):
        if lag >= series.shape[1] - 1:
            real_acf.append(0.0)
        else:
            x_t = series[:, :-lag-1].flatten()
            x_k = series[:, lag+1:].flatten()
            real_acf.append(np.corrcoef(x_t, x_k)[0, 1])
    ax.plot(range(max_lag), real_acf, color='black', marker='o', label='Real')

    for method in methods:
        series = generated_data[method][:len(sanity_subset), :, 0]
        acf_vals = []
        for lag in range(max_lag):
            if lag >= series.shape[1] - 1:
                acf_vals.append(0.0)
            else:
                x_t = series[:, :-lag-1].flatten()
                x_k = series[:, lag+1:].flatten()
                acf_vals.append(np.corrcoef(x_t, x_k)[0, 1])
        ax.plot(range(max_lag), acf_vals, marker='x', label=method)
    ax.set_title('ACF Plot')
    ax.set_xlabel('Lag')
    ax.legend(fontsize=8)

    fig.suptitle('Sanity Checks On Shared Evaluation Subset', fontsize=18, y=1.02)
    plt.tight_layout()
    plt.show()

    if stress_trajectories:
        for model_name, stress in stress_trajectories.items():
            plt.figure(figsize=(10, 4))
            plt.plot(np.mean(stress, axis=0), linewidth=2)
            plt.title(f'Stress Factor Mean Trajectory: {model_name}')
            plt.xlabel('Time Step')
            plt.ylabel('Stress')
            plt.show()
